# WikiEath

**氏名：［兼平祥汰］**

AIの利用：Claudeに、コードの下書きと、Notebookの構成づくりを手伝ってもらった。

## 調べたいこと

「東京」「柏まつり」「地球」のWikipedia記事には、どんな場所の情報が付いているのかを表示させる。

出典：[Wikipedia REST API（page/summary）](https://ja.wikipedia.org/api/rest_v1/)／取得日：2026年10月21日／日本語版Wikipedia

以下のコードは、3つの記事の要約と座標を取得します。追加インストールは不要です。

In [1]:
import json
from urllib.parse import quote
from urllib.request import Request, urlopen

# 調べる3つの言葉。title は記事名、fb は記事に座標がないときの予備の座標
PRESETS = {
    "東京": {"title": "東京都", "fb": (35.6895, 139.6917)},
    "柏まつり": {"title": "柏まつり", "fb": (35.8618, 139.9710)},
    "地球": {"title": "地球", "globe": True},
}

results = {}

for key, p in PRESETS.items():
    url = "https://ja.wikipedia.org/api/rest_v1/page/summary/" + quote(p["title"])
    request = Request(url, headers={"User-Agent": "week03-homework (university class)"})
    with urlopen(request, timeout=20) as response:
        data = json.load(response)

    # 場所を決める: 地球は全体、ほかは記事の座標（なければ予備の座標）
    if p.get("globe"):
        lat, lon, how = None, None, "地球全体"
    elif "coordinates" in data:
        lat, lon, how = data["coordinates"]["lat"], data["coordinates"]["lon"], "記事の座標"
    else:
        (lat, lon), how = p["fb"], "予備の座標（記事に座標なし）"

    results[key] = {
        "title": data["title"],
        "description": data.get("description", ""),
        "extract": data.get("extract", ""),
        "lat": lat,
        "lon": lon,
        "how": how,
        "url": data["content_urls"]["desktop"]["page"],
    }

# 提出用に、取得したデータをJSONとして保存する
with open("wikipedia-globe.json", "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=1)

## Wikipediaの記事を取り出す

`results`は辞書で、`"東京"`・`"柏まつり"`・`"地球"`のそれぞれに、記事の情報が入っている。

その中は辞書で、`title`（記事名）、`extract`（要約）、`lat`と`lon`（緯度と経度）、`how`（場所の決め方）を持ちます。地球は特定の場所がないので、`lat`と`lon`は`None`です。

In [2]:
for key, r in results.items():
    print(f"【{key}】 記事名: {r['title']}")
    print(f"  説明: {r['description']}")
    if r["lat"] is None:
        print(f"  場所: {r['how']}")
    else:
        print(f"  場所: 北緯{r['lat']:.3f}、東経{r['lon']:.3f}（{r['how']}）")
    print(f"  要約の文字数: {len(r['extract'])} 字")
    print(f"  要約の最初: {r['extract'][:50]}…")
    print()

【東京】 記事名: 東京都
  説明: 日本国の首都、日本国の都道府県の1つ
  場所: 北緯35.689、東経139.692（記事の座標）
  要約の文字数: 77 字
  要約の最初: 東京都（とうきょうと、 は、日本の関東地方に位置する都。都庁所在地は新宿区、条例上の位置づけは新宿区…

【柏まつり】 記事名: 柏まつり
  説明: 
  場所: 北緯35.862、東経139.971（予備の座標（記事に座標なし））
  要約の文字数: 75 字
  要約の最初: 柏まつり（かしわまつり）とは、千葉県柏市で、毎年7月下旬に行われる祭り。毎年70万人の人出がある、柏…

【地球】 記事名: 地球
  説明: 太陽系第三惑星
  場所: 地球全体
  要約の文字数: 113 字
  要約の最初: 地球 は太陽系の惑星の1つ。水星、金星に次いで太陽から3番目に近いため太陽系第3惑星と言われている。…



## 地球儀で見る

下のセルを実行すると、地球儀が出ます。ボタンを押すと、その記事の場所へ回転して、周辺が明るく光ります。地球は全体が光ります。ドラッグでも回せます。

In [3]:
from IPython.display import HTML, display

MASK = "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOAfAP8HAAAAAAAAAAAAAAAAAAAAAADo//z//wcAAAAAAAACAAAAAAAAAAAAhvvw//8PAPABAAAAwAMAAAAAAAAAwADkw////wEABAAAAABgAAAAAAAAAADAUT8A/v8PAAAAAAwA/j8AuAEAAAAAcBHtDcD/fwAAAAAwAPz/fwMAABCAAQD5w/wD8P8FAAAcAIL7//9//xOAAP/ff0668QD/HwAA+A8At///////v3/w/////x8++B8AAOD/1//7////////jP////+/+AE/gAcAn9f//////////w/w/////wEs4AEAAHz+////////////gL////8HeAAcAADg5///////////9ADgAf7/f4AnAAAAAH78////////H0QAAAiA//8f8AcAAIBB4////////38ADwAQAOD//5//AQAAHAT/////////A3AAAAAA/v//+T8AAGDz//////////8DAQAAAMD/////AwAAsP//////////LwAAAAAA6P///2IAAAD+//////////8CAAAAAAD///8/CAAA4P//////////JwAAAAAA8P///wYAAAD+/unz/////z8AAAAAAAD///8HAAAA/pgPPP//////MQAAAAAA8P//PwAAAMBD9v7n/////wcBAAAAAAD///8AAAAAPkD7f/7///8hEAAAAAAA4P//DwAAAIDhAv/n////f8YAAAAAAAD8//8AAAAA+AdE//////8jDwAAAAAAgP//AwAAAMD/APD/////PxgAAAAAAADw/x8AAAAA/n/v//////8HAAAAAAAAAPwDAgAAAOD////7////fwAAAAAAAACgHyAAAACA//9/f/7///8DAAAAAAAAAPQBAAAAAPj//+cv+P//PwAAAAAAAAAAHjAAAADA/////g/+//8EAAAAAAAAAOBhCAAAAP7//99/4D//AAAAAAAAAAAAPAMEAADA////+Qf84BcAAAAAAAAAAAA/AAAAAPz//58fgAf+QAAAAAAAAAAAAA8AAADg////ewA4gA8EAAAAAAAAAADAAAAAAPz//38BgAP4QQAAAAAAAAAAAAgPAADA////zwAwgAwQAAAAAAAAAAAA9Q8AAPj///8HAAVIAAAAAAAAAAAAAID/AQAA////fwBAAAAQAAAAAAAAAAAA+P8AAGDh//8DAAA0GAAAAAAAAAAAAID/HwAAAPj/HwAAgMIBAAAAAAAAAAAA/P8BAACA//8AAAAYXgAAAAAAAAAAAMD/fwAAAPz/BwAAAOOBAQAAAAAAAAAA/P8/AACA/z8AAABgbtQBAAAAAAAAAOD//w8AAPD/AwAAAAQIeAAAAAAAAAAA/P//AQAA/z8AAACAA4APAQAAAAAAAID//w8AAPD/AwAAAAARsEAAAAAAAAAA+P9/AAAA/j8AAAAAAAAAAAAAAAAAAAD//wcAAPD/QwAAAACAIwAAAAAAAAAA8P9/AAAA/z8EAAAAAD8GIAAAAAAAAAD8/wMAAPD/cQAAAAD4ZwAAAAAAAAAAgP8/AAAA/w8HAAAAgP8HAAAAAAAAAAD4/wMAAOD/MAAAAAD//wEBAAAAAAAAgP8PAAAA/g8DAAAA+P8fAAAAAAAAAAD4PwAAAOB/EAAAAID//wMAAAAAAAAAgP8DAAAA/AMAAAAA+P9/AAAAAAAAAAD8HwAAAMA/AAAAAID//wcAAAAAAAAAwP8BAAAA+AEAAAAA8P9/AAAAAAAAAAD8DwAAAIAPAAAAAAAP/gMAAAAAAAAAwB8AAAAAAAAAAAAAEIAfAAEAAAAAAAD+AwAAAAAAAAAAAAAA8AEgAAAAAAAA4AcAAAAAAAAAAAAAAAAAAAYAAAAAAABeAAAAAAAAAAAAAAAAwAAwAAAAAAAAwAMAAAAAAAAAAAAAAAAIgAEAAAAAAAAeAAAAAAAAAAAAAAAAAAAMAAAAAAAA4AEAAAAAAAAAAAAAAAAAAAAAAAAAAAAPAAAAAAAAAAABAAAAAAAAAAAAAAAAcAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOAAAAAAAAAAAAAAAAAAAAAAAAAAAAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMAAAAAAAB4AEDwn/8HAAAAAAAAAAAwAAAAAACA/P/h/////w8AAAAAAAAAwAcAAAD4////z///////PwAAAAAAHALwAACA//////////////8HAADw/y///wMAAP7/////////////HwAA+P///38AAID///////////////8AAPL/////BwAO////////////////DwAA8P////8HEPD//////////////z8AAOD/////////////////////////H/AfwP//////////////////////////////////////////////////////////////////////////////////////"  # 陸地を2度ごとの点で表した地図データ（Natural Earth 110mの陸地から作成）

TEMPLATE = r'''
(function(){
const ID="__ID__",DATA=__DATA__,MASK="__MASK__";
const root=document.getElementById(ID);
const cv=root.querySelector('canvas'),cx=cv.getContext('2d');
const msg=root.querySelector('.msg'),card=root.querySelector('.card');
const W=180,H=90,rad=Math.PI/180;
const bin=atob(MASK),land=[];
for(let j=0;j<H;j++)for(let i=0;i<W;i++){const k=j*W+i;if(bin.charCodeAt(k>>3)>>(k&7)&1)land.push([90-(j+.5)*2,-180+(i+.5)*2]);}
const unit=(la,lo)=>[Math.cos(la*rad)*Math.sin(lo*rad),Math.sin(la*rad),Math.cos(la*rad)*Math.cos(lo*rad)];
const pts=land.map(([la,lo])=>unit(la,lo));
const w=cv.width,h=cv.height,R=Math.min(w,h)*.40,X0=w/2,Y0=h/2;
let rotLon=140,rotLat=20,target=null,auto=true,hl=null,amt=0;
function proj(la,lo){const a=la*rad,b=(lo-rotLon)*rad,c=rotLat*rad;
  const x=Math.cos(a)*Math.sin(b),y=Math.sin(a),z=Math.cos(a)*Math.cos(b);
  return [x,y*Math.cos(c)-z*Math.sin(c),y*Math.sin(c)+z*Math.cos(c)];}
function draw(t){
  amt+=((hl?1:0)-amt)*.05;
  const pulse=(Math.sin(t/450)+1)/2,eb=hl&&hl.globe?amt:0;
  cx.clearRect(0,0,w,h);
  const g=cx.createRadialGradient(X0,Y0,R*.9,X0,Y0,R*(1.35+eb*.25));
  g.addColorStop(0,'rgba('+(79+eb*150|0)+','+(209+eb*30|0)+',255,'+(.16+eb*.35)+')');g.addColorStop(1,'rgba(79,209,255,0)');
  cx.fillStyle=g;cx.beginPath();cx.arc(X0,Y0,R*1.6,0,7);cx.fill();
  cx.fillStyle='#07101e';cx.beginPath();cx.arc(X0,Y0,R,0,7);cx.fill();
  for(let i=0;i<land.length;i++){
    const[p,q,z]=proj(land[i][0],land[i][1]);if(z<=0)continue;
    let a=.25+z*.6,s=2,r=120,gr=200,b=255;
    if(hl){let k;
      if(hl.globe)k=.55+.45*pulse;
      else{const d=pts[i][0]*hl.u[0]+pts[i][1]*hl.u[1]+pts[i][2]*hl.u[2];k=Math.exp(-Math.pow(Math.acos(Math.max(-1,Math.min(1,d)))/.22,2));}
      a=a*(1-.55*amt*(1-k))+k*amt*.6;s=2+k*amt*2.2;r=120+k*amt*135;gr=200+k*amt*55;b=255-k*amt*120;}
    cx.fillStyle='rgba('+(r|0)+','+(gr|0)+','+(b|0)+','+Math.min(1,a)+')';
    cx.fillRect(X0+p*R-s/2,Y0-q*R-s/2,s,s);
  }
  cx.strokeStyle='rgba('+(79+eb*150|0)+',209,255,'+(.35+eb*.5)+')';cx.lineWidth=1.2+eb*1.5;cx.beginPath();cx.arc(X0,Y0,R,0,7);cx.stroke();
  if(hl&&!hl.globe){const[p,q,z]=proj(hl.lat,hl.lon);
    if(z>0){const X=X0+p*R,Y=Y0-q*R,gr=R*.32*amt;
      const g2=cx.createRadialGradient(X,Y,0,X,Y,gr);
      g2.addColorStop(0,'rgba(255,235,170,'+(.75*amt)+')');g2.addColorStop(.4,'rgba(255,190,90,'+(.25*amt)+')');g2.addColorStop(1,'rgba(255,170,60,0)');
      cx.fillStyle=g2;cx.beginPath();cx.arc(X,Y,gr,0,7);cx.fill();
      cx.fillStyle='#fff';cx.beginPath();cx.arc(X,Y,4.5,0,7);cx.fill();
      for(let n=0;n<2;n++){const ph=((t/1400)+n*.5)%1;cx.strokeStyle='rgba(255,220,140,'+((1-ph)*.8*amt)+')';cx.lineWidth=1.5;cx.beginPath();cx.arc(X,Y,6+ph*34,0,7);cx.stroke();}}}
  if(target){let d=target.lon-rotLon;d=((d+540)%360)-180;rotLon+=d*.06;rotLat+=(target.lat-rotLat)*.06;
    if(Math.abs(d)<.05&&Math.abs(target.lat-rotLat)<.05)target=null;}
  else if(auto)rotLon+=.06;
  requestAnimationFrame(draw);
}
requestAnimationFrame(draw);
let drag=null;
cv.addEventListener('pointerdown',e=>{drag=[e.clientX,e.clientY];auto=false;target=null;cv.setPointerCapture(e.pointerId)});
cv.addEventListener('pointermove',e=>{if(!drag)return;rotLon-=(e.clientX-drag[0])*.3;rotLat=Math.max(-85,Math.min(85,rotLat+(e.clientY-drag[1])*.3));drag=[e.clientX,e.clientY]});
cv.addEventListener('pointerup',()=>{drag=null});
function show(key){
  const d=DATA[key];if(!d)return;
  root.querySelectorAll('button').forEach(b=>b.style.background=b.textContent===key?'#4fd1ff':'#0d1a2e');
  root.querySelectorAll('button').forEach(b=>b.style.color=b.textContent===key?'#05080f':'#dce8ff');
  if(d.lat==null){hl={globe:true};target=null;auto=true;}
  else{hl={lat:d.lat,lon:d.lon,u:unit(d.lat,d.lon)};amt=0;target={lat:d.lat,lon:d.lon};auto=false;}
  card.style.display='block';
  card.querySelector('.t').textContent=d.title;
  card.querySelector('.d').textContent=d.description||'';
  card.querySelector('.w').textContent='場所: '+d.how;
  card.querySelector('.x').textContent=d.extract||'';
  card.querySelector('a').href=d.url;
  msg.textContent=d.lat==null?'地球全体を光らせています。':d.title+' の場所へ回転しました。';
}
root.querySelectorAll('button').forEach(b=>b.onclick=()=>show(b.textContent));
})();
'''

BODY = '''
<div id="__ID__" style="position:relative;width:760px;max-width:100%;background:#05080f;border-radius:12px;overflow:hidden;font-family:sans-serif;color:#dce8ff">
  <canvas width="760" height="460" style="display:block;width:100%;cursor:grab;touch-action:none"></canvas>
  <div style="position:absolute;top:12px;left:12px">
    <button style="padding:6px 14px;margin-right:6px;border-radius:999px;border:1px solid #17233a;background:#0d1a2e;color:#dce8ff;cursor:pointer">東京</button><button style="padding:6px 14px;margin-right:6px;border-radius:999px;border:1px solid #17233a;background:#0d1a2e;color:#dce8ff;cursor:pointer">柏まつり</button><button style="padding:6px 14px;border-radius:999px;border:1px solid #17233a;background:#0d1a2e;color:#dce8ff;cursor:pointer">地球</button>
    <div class="msg" style="margin-top:8px;font-size:12px;color:#7a8cab">ボタンを押してください。</div>
  </div>
  <div class="card" style="display:none;position:absolute;right:12px;bottom:12px;width:300px;max-height:75%;overflow:auto;background:rgba(10,18,32,.9);border:1px solid #17233a;border-radius:12px;padding:12px 14px">
    <div class="t" style="font-size:17px;font-weight:600"></div>
    <div class="d" style="color:#4fd1ff;font-size:12px;margin:2px 0 6px"></div>
    <div class="w" style="color:#ffd37a;font-size:12px;margin-bottom:6px"></div>
    <div class="x" style="font-size:12px;line-height:1.7;color:#b8c7e3;margin-bottom:8px"></div>
    <a target="_blank" rel="noopener" style="color:#4fd1ff;font-size:12px">Wikipediaで読む</a>
  </div>
</div>
<script>__JS__</script>
'''

uid = "globe-" + str(abs(hash(json.dumps(results, sort_keys=True))))
js = TEMPLATE.replace("__ID__", uid).replace("__DATA__", json.dumps(results, ensure_ascii=False)).replace("__MASK__", MASK)
display(HTML(BODY.replace("__ID__", uid).replace("__JS__", js)))

## 分かったこと

「東京」の記事は、記事の座標を参照して北緯35.689、東経139.692の場所を地球儀上で示せるように出来た。しかし、「柏まつり」の記事は、座標のデータがなかったので、東京の座標と同じにした。このことから、APIを用いてデータを取得したときに、データがないということもデータの一つであることが分かった。また、「地球」は特定の場所を持たず、地球全体を光らせた。

地球儀では、東京と柏まつりは同じ場所のように見えた。

**確認：** 元のJSON（`wikipedia-globe.json`）の`lat`と`lon`を開き、出力された緯度経度と同じことを確かめた。また、地球儀で光った場所が、地図の位置と合っていることを目で見て確かめた。

**限界：** 今回はWikipediaのAPIを使ったが、検索欄に関して　　　地球儀に表示させるにはAPIを使って座標を取得する必要がある。しかし、今回座標は記事に付いた1点だけで、まつりの会場や都の範囲そのものではない。記事に座標がないときは、私が決めた予備の座標を使っている。陸地の点は2度ごとなので、細かい形は出ない。記事は書き換えられるので、取得日が変わると値も変わってしまう。再実行したら観察も書き直す。

## 通信できないときだけ

保存データ`wikipedia-globe.json`を同じフォルダに置き、最初のCode Cellを次のコードに置き換えます。2026年10月21日の保存例なので、最新のデータではありません。

```python
import json

with open("wikipedia-globe.json", encoding="utf-8") as file:
    results = json.load(file)
```

## 自分の課題として提出する

自分の問いでAPIデータを読み、何か一つ見せよう。文章の出力だけでもOK。発見・確かめた値・限界・出典URLと取得日を短く書きます。AIを使った場合は、用途を一言添えてください。

**一番上にタイトルと氏名。全Cellを実行し、出力と観察を保存。**

10月21日（水）23:59 JSTまでに、`.ipynb`と必要なファイルをUNIPAへ。